# Prepare Supplementary Analyses


In [ ]:
from __future__ import annotations

import ast
import json
import os
import re
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import ImageColor
import plotly.graph_objects as go
import textstat

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RAW_DATA_DIR = DATA_DIR / "raw" / "datasets"

REPORT_INDEX_PATH = PROJECT_ROOT / "report_ids.csv"
REPORT_ID_COLUMN = "report_id"
COMPANIES_PATH = RAW_DATA_DIR / "companies.csv"
INDICATORS_PATH = PROCESSED_DATA_DIR / "esg_indicators_postprocessed.csv"
INDICATOR_METADATA_PATH = RAW_DATA_DIR / "indicator_metadata.csv"

TEXT_DIR = PROCESSED_DATA_DIR / "reports_txt"
VECTORSTORE_DIR = PROCESSED_DATA_DIR / "vectorstores"

MIN_TEXT_CHARACTERS = 5000

HEATMAP_OUTPUT_DIR = NOTEBOOK_DIR
INDICATOR_EXTRACTABILITY_OUTPUT_PATH = NOTEBOOK_DIR / "indicator_extractability_analysis.csv"
LATEX_OUTPUT_PATH = NOTEBOOK_DIR / "indicator_coverage_longtable.tex"
TOP_QUALITY_DOCUMENTS_OUTPUT_PATH = PROCESSED_DATA_DIR / "top_quality_documents_yearly.json"
CONSTANT_REPORTERS_OUTPUT_PATH = PROCESSED_DATA_DIR / "constant_reporters.json"


## Helpers


In [ ]:
def parse_report_id(report_id: str) -> tuple[Optional[str], Optional[int], Optional[str]]:
    parts = str(report_id).rsplit("_", 2)
    if len(parts) != 3:
        return None, None, None

    company_id, year_text, report_type = parts
    try:
        year = int(year_text)
    except ValueError:
        return None, None, None
    return company_id, year, report_type


def load_report_index(report_index_path: Path, report_id_column: str = REPORT_ID_COLUMN) -> pd.DataFrame:
    report_index_df = pd.read_csv(report_index_path)
    if report_id_column not in report_index_df.columns:
        raise ValueError(f"Missing required column: {report_id_column}")

    parsed = report_index_df[report_id_column].astype(str).apply(parse_report_id)
    report_index_df[["firm", "year", "report_type"]] = pd.DataFrame(parsed.tolist(), index=report_index_df.index)

    report_index_df = report_index_df.dropna(subset=["firm", "year"]).copy()
    report_index_df["year"] = report_index_df["year"].astype(int)
    report_index_df["report_id"] = report_index_df[report_id_column].astype(str)
    return report_index_df[["report_id", "firm", "year", "report_type"]]


def build_report_grid(report_index_df: pd.DataFrame) -> pd.DataFrame:
    firms = np.sort(report_index_df["firm"].dropna().unique())
    years = np.sort(report_index_df["year"].dropna().astype(int).unique())
    population_df = pd.MultiIndex.from_product([firms, years], names=["firm", "year"]).to_frame(index=False)
    return population_df.merge(report_index_df, on=["firm", "year"], how="left")


def candidate_text_paths(text_dir: Path, firm: str, year: int) -> list[Path]:
    return [
        text_dir / f"{firm}_{year}.json",
        text_dir / f"{firm}-{year}.json",
        text_dir / f"{firm}_{year}.txt",
        text_dir / f"{firm}-{year}.txt",
    ]


def load_text_artifact(path: Path) -> Optional[str]:
    suffix = path.suffix.lower()
    if suffix == ".txt":
        try:
            return path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            return None

    if suffix == ".json":
        try:
            with path.open("r", encoding="utf-8", errors="replace") as handle:
                payload = json.load(handle)
        except (OSError, json.JSONDecodeError):
            return None

        if isinstance(payload, dict):
            for key in ["text", "full_text", "content"]:
                value = payload.get(key)
                if isinstance(value, str):
                    return value
        return None

    return None


def has_usable_text(text_dir: Path, firm: str, year: int, min_characters: int = MIN_TEXT_CHARACTERS) -> bool:
    for path in candidate_text_paths(text_dir, firm, year):
        if not path.is_file():
            continue
        text = load_text_artifact(path)
        if isinstance(text, str) and len(text.strip()) >= min_characters:
            return True
    return False


def candidate_vectorstore_paths(vectorstore_dir: Path, firm: str, year: int) -> list[Path]:
    return [
        vectorstore_dir / f"{firm}_{year}.vectorstore",
        vectorstore_dir / f"{firm}-{year}.vectorstore",
    ]


def has_usable_vectorstore(vectorstore_dir: Path, firm: str, year: int) -> bool:
    for vectorstore_path in candidate_vectorstore_paths(vectorstore_dir, firm, year):
        if vectorstore_path.is_dir() and (vectorstore_path / "index.faiss").is_file() and (vectorstore_path / "index.pkl").is_file():
            return True
    return False


def build_firm_stage_table(
    report_grid_df: pd.DataFrame,
    text_dir: Path = TEXT_DIR,
    vectorstore_dir: Path = VECTORSTORE_DIR,
    min_text_characters: int = MIN_TEXT_CHARACTERS,
) -> pd.DataFrame:
    firm_year_df = report_grid_df[["firm", "year"]].drop_duplicates().sort_values(["firm", "year"])

    artifact_rows = []
    for row in firm_year_df.itertuples(index=False):
        artifact_rows.append(
            {
                "firm": row.firm,
                "year": int(row.year),
                "text_extracted": has_usable_text(
                    text_dir,
                    row.firm,
                    int(row.year),
                    min_characters=min_text_characters,
                ),
                "chunks_present": has_usable_vectorstore(vectorstore_dir, row.firm, int(row.year)),
            }
        )

    artifact_df = pd.DataFrame(artifact_rows)
    stage_df = report_grid_df.merge(artifact_df, on=["firm", "year"], how="left")
    stage_df["report_available"] = stage_df["report_id"].notna()
    stage_df["text_extracted"] = stage_df["text_extracted"].astype("boolean").fillna(False).astype(bool)
    stage_df["chunks_present"] = stage_df["chunks_present"].astype("boolean").fillna(False).astype(bool)

    return stage_df[
        [
            "firm",
            "year",
            "report_id",
            "report_available",
            "text_extracted",
            "chunks_present",
        ]
    ].sort_values(["firm", "year", "report_id"], na_position="last").reset_index(drop=True)


def collapse_firm_year_stage_data(firm_stage_df: pd.DataFrame) -> pd.DataFrame:
    return (
        firm_stage_df.groupby(["firm", "year"], as_index=False)
        .agg(
            report_available=("report_available", "any"),
            text_extracted=("text_extracted", "any"),
            chunks_present=("chunks_present", "any"),
            n_reports_available=("report_available", "sum"),
        )
    )


def to_rgba(color: str, alpha: float = 0.35) -> str:
    red, green, blue = ImageColor.getrgb(color)
    return f"rgba({red},{green},{blue},{alpha})"


def plot_representativeness_heatmap(
    companies_df: pd.DataFrame,
    firm_stage_df: pd.DataFrame,
    *,
    value: str = "coverage_rate",
    by: str = "sector",
    figsize: tuple[float, float] = (9, 5),
    heat_ax_pos: tuple[float, float, float, float] = (0.28, 0.15, 0.58, 0.75),
    cbar_ax_pos: tuple[float, float, float, float] = (0.88, 0.15, 0.03, 0.75),
    tick_labelsize: int = 8,
    output_path: Optional[Path] = None,
) -> tuple[plt.Figure, plt.Axes]:
    required_company_columns = {"firm", "country", "primary_sics_sector", "year"}
    missing_company_columns = required_company_columns - set(companies_df.columns)
    if missing_company_columns:
        raise ValueError(f"Missing company columns: {missing_company_columns}")

    required_stage_columns = {"firm", "year", "report_available"}
    missing_stage_columns = required_stage_columns - set(firm_stage_df.columns)
    if missing_stage_columns:
        raise ValueError(f"Missing stage columns: {missing_stage_columns}")

    population_df = companies_df[["firm", "year", "country", "primary_sics_sector"]].drop_duplicates().copy()
    population_df["year"] = population_df["year"].astype(int)
    population_df = population_df.dropna(subset=["primary_sics_sector"])

    collapsed_stage_df = collapse_firm_year_stage_data(firm_stage_df)
    merged_df = population_df.merge(
        collapsed_stage_df[["firm", "year", "report_available", "n_reports_available"]].rename(
            columns={"report_available": "any_available"}
        ),
        on=["firm", "year"],
        how="left",
    )
    merged_df["any_available"] = merged_df["any_available"].astype("boolean").fillna(False).astype(bool)
    merged_df["n_reports_available"] = merged_df["n_reports_available"].fillna(0).astype(int)

    if by.lower() == "sector":
        group_column = "primary_sics_sector"
        y_label = "Industry"
    elif by.lower() == "country":
        group_column = "country"
        y_label = "Country"
    else:
        raise ValueError("`by` must be either 'sector' or 'country'.")

    if value == "coverage_rate":
        heatmap_df = (
            merged_df.groupby([group_column, "year"], as_index=False)
            .agg(metric=("any_available", "mean"))
        )
        colorbar_label = "Coverage rate"
        vmin, vmax = 0.0, 1.0
    elif value == "n_reports":
        heatmap_df = (
            merged_df.groupby([group_column, "year"], as_index=False)
            .agg(metric=("n_reports_available", "sum"))
        )
        colorbar_label = "Available reports"
        vmin, vmax = None, None
    else:
        raise ValueError("`value` must be either 'coverage_rate' or 'n_reports'.")

    pivot_df = (
        heatmap_df.pivot(index=group_column, columns="year", values="metric")
        .sort_index()
        .reindex(sorted(heatmap_df["year"].unique()), axis=1)
    )

    fig = plt.figure(figsize=figsize)
    heatmap_ax = fig.add_axes(heat_ax_pos)
    colorbar_ax = fig.add_axes(cbar_ax_pos)

    image = heatmap_ax.imshow(pivot_df.values, aspect="auto", cmap="Blues", vmin=vmin, vmax=vmax)
    heatmap_ax.set_xlabel("Year")
    heatmap_ax.set_ylabel(y_label)
    heatmap_ax.set_xticks(np.arange(pivot_df.shape[1]))
    heatmap_ax.set_xticklabels(pivot_df.columns.astype(str), fontsize=tick_labelsize)
    heatmap_ax.set_yticks(np.arange(pivot_df.shape[0]))
    heatmap_ax.set_yticklabels(pivot_df.index.astype(str), fontsize=tick_labelsize)

    colorbar = fig.colorbar(image, cax=colorbar_ax)
    colorbar.set_label(colorbar_label)

    missing_mask = np.isnan(pivot_df.values)
    if missing_mask.any():
        for row_index, column_index in zip(*np.where(missing_mask)):
            heatmap_ax.add_patch(
                plt.Rectangle(
                    (column_index - 0.5, row_index - 0.5),
                    1,
                    1,
                    fill=True,
                    color="white",
                    ec="lightgray",
                    lw=0.5,
                )
            )

    plt.show()
    if output_path is not None:
        fig.savefig(output_path, format=output_path.suffix.lstrip("."), bbox_inches="tight")
    return fig, heatmap_ax


def plot_firm_year_pipeline_sankey(
    firm_stage_df: pd.DataFrame,
    *,
    title: str = "Firm-year coverage across the pipeline",
    height: int = 350,
    width: int = 980,
) -> go.Figure:
    collapsed_stage_df = collapse_firm_year_stage_data(firm_stage_df)

    report_ok = collapsed_stage_df["report_available"].fillna(False)
    text_ok = report_ok & collapsed_stage_df["text_extracted"].fillna(False)
    chunk_ok = text_ok & collapsed_stage_df["chunks_present"].fillna(False)

    total_count = int(len(collapsed_stage_df))
    report_count = int(report_ok.sum())
    no_report_count = total_count - report_count
    text_count = int(text_ok.sum())
    no_text_count = report_count - text_count
    chunk_count = int(chunk_ok.sum())
    no_chunk_count = text_count - chunk_count

    node_definitions = {
        "all": {
            "label": f"<b>Company-year observations</b><br>{total_count:,}",
            "x": 0.01,
            "y": 0.50,
            "color": "cornflowerblue",
        },
        "report": {
            "label": f"<b>Report retrieved</b><br>{report_count:,}",
            "x": 0.33,
            "y": 0.30,
            "color": "cadetblue",
        },
        "no_report": {
            "label": f"<b>No report retrieved</b><br>{no_report_count:,}",
            "x": 0.33,
            "y": 0.70,
            "color": "darkgrey",
        },
        "text": {
            "label": f"<b>Text extracted</b><br>{text_count:,}",
            "x": 0.66,
            "y": 0.30,
            "color": "cadetblue",
        },
        "no_text": {
            "label": f"<b>No usable text</b><br>{no_text_count:,}",
            "x": 0.66,
            "y": 0.70,
            "color": "darkgrey",
        },
        "chunks": {
            "label": f"<b>Chunking successful</b><br>{chunk_count:,}",
            "x": 0.99,
            "y": 0.30,
            "color": "darkseagreen",
        },
        "no_chunks": {
            "label": f"<b>No usable chunks</b><br>{no_chunk_count:,}",
            "x": 0.99,
            "y": 0.70,
            "color": "darkgrey",
        },
    }

    link_definitions = [
        ("all", "report", report_count),
        ("all", "no_report", no_report_count),
        ("report", "text", text_count),
        ("report", "no_text", no_text_count),
        ("text", "chunks", chunk_count),
        ("text", "no_chunks", no_chunk_count),
    ]
    link_definitions = [link for link in link_definitions if link[2] > 0]

    node_order = ["all", "report", "no_report", "text", "no_text", "chunks", "no_chunks"]
    used_nodes = {node for source, target, _ in link_definitions for node in (source, target)}
    nodes = [node for node in node_order if node in used_nodes]
    node_index = {node: index for index, node in enumerate(nodes)}

    figure = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    label=[node_definitions[node]["label"] for node in nodes],
                    x=[node_definitions[node]["x"] for node in nodes],
                    y=[node_definitions[node]["y"] for node in nodes],
                    pad=22,
                    thickness=18,
                    color=[node_definitions[node]["color"] for node in nodes],
                ),
                link=dict(
                    source=[node_index[source] for source, _, _ in link_definitions],
                    target=[node_index[target] for _, target, _ in link_definitions],
                    value=[value for _, _, value in link_definitions],
                    color=[
                        to_rgba(
                            "darkgrey" if target.startswith("no_") else node_definitions[source]["color"],
                            0.35,
                        )
                        for source, target, _ in link_definitions
                    ],
                    hovertemplate="Flow: %{value:,}<extra></extra>",
                ),
            )
        ]
    )

    figure.update_layout(
        title=title,
        hovermode="x",
        font=dict(size=15),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=height,
        width=width,
        margin=dict(l=20, r=20, t=50, b=50),
    )
    figure.show()
    return figure


def parse_model_output(model_output: object) -> Optional[dict[str, Any]]:
    if model_output is None or (isinstance(model_output, float) and pd.isna(model_output)):
        return None
    if isinstance(model_output, dict):
        return model_output
    if not isinstance(model_output, str) or not model_output.strip():
        return None

    payload = model_output.strip()
    try:
        parsed = json.loads(payload)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    try:
        parsed = ast.literal_eval(payload)
    except (SyntaxError, ValueError):
        return None
    return parsed if isinstance(parsed, dict) else None


def model_output_has_value(model_output: object) -> bool:
    parsed = parse_model_output(model_output)
    if not isinstance(parsed, dict):
        return False

    value = parsed.get("value")
    if value is None:
        return False
    if isinstance(value, str) and not value.strip():
        return False
    return True


def prepare_indicator_observations(
    firm_stage_df: pd.DataFrame,
    indicator_df: pd.DataFrame,
    *,
    firm_col_firm_stage: str = "firm",
    year_col_firm_stage: str = "year",
    chunks_col_firm_stage: str = "chunks_present",
    firm_col_indicator: str = "company_id",
    year_col_indicator: str = "year",
    indicator_col_indicator: str = "data_point_id",
    model_output_col: str = "model_output",
    value_final_col: str = "value_final",
    indicator_ids: Optional[list[str]] = None,
) -> tuple[pd.DataFrame, pd.Index, pd.DataFrame]:
    collapsed_stage_df = collapse_firm_year_stage_data(firm_stage_df)
    eligible_firm_years_df = collapsed_stage_df.loc[
        collapsed_stage_df[chunks_col_firm_stage].fillna(False),
        [firm_col_firm_stage, year_col_firm_stage],
    ].copy()
    eligible_firm_years_df[year_col_firm_stage] = eligible_firm_years_df[year_col_firm_stage].astype(int)

    if indicator_ids is None:
        indicator_ids = sorted(indicator_df[indicator_col_indicator].dropna().astype(str).unique().tolist())
    indicator_index = pd.Index(indicator_ids, name=indicator_col_indicator)

    observations_df = indicator_df[
        [firm_col_indicator, year_col_indicator, indicator_col_indicator, model_output_col, value_final_col]
    ].copy()
    observations_df = observations_df[
        observations_df[indicator_col_indicator].astype(str).isin(indicator_index.astype(str))
    ].copy()
    observations_df[year_col_indicator] = observations_df[year_col_indicator].astype(int)
    observations_df[indicator_col_indicator] = observations_df[indicator_col_indicator].astype(str)
    observations_df["extracted_present"] = observations_df[model_output_col].apply(model_output_has_value)
    observations_df["final_present"] = observations_df[value_final_col].notna()

    eligible_lookup_df = eligible_firm_years_df.rename(
        columns={
            firm_col_firm_stage: firm_col_indicator,
            year_col_firm_stage: year_col_indicator,
        }
    )
    observations_df = observations_df.merge(
        eligible_lookup_df,
        on=[firm_col_indicator, year_col_indicator],
        how="inner",
    )

    observations_df = (
        observations_df.groupby(
            [firm_col_indicator, year_col_indicator, indicator_col_indicator],
            as_index=False,
        )[["extracted_present", "final_present"]]
        .any()
    )
    return eligible_firm_years_df, indicator_index, observations_df


def compute_indicator_opportunity_counts(
    firm_stage_df: pd.DataFrame,
    indicator_df: pd.DataFrame,
    *,
    firm_col_firm_stage: str = "firm",
    year_col_firm_stage: str = "year",
    chunks_col_firm_stage: str = "chunks_present",
    firm_col_indicator: str = "company_id",
    year_col_indicator: str = "year",
    indicator_col_indicator: str = "data_point_id",
    model_output_col: str = "model_output",
    value_final_col: str = "value_final",
    indicator_ids: Optional[list[str]] = None,
) -> dict[str, int]:
    eligible_firm_years_df, indicator_index, observations_df = prepare_indicator_observations(
        firm_stage_df,
        indicator_df,
        firm_col_firm_stage=firm_col_firm_stage,
        year_col_firm_stage=year_col_firm_stage,
        chunks_col_firm_stage=chunks_col_firm_stage,
        firm_col_indicator=firm_col_indicator,
        year_col_indicator=year_col_indicator,
        indicator_col_indicator=indicator_col_indicator,
        model_output_col=model_output_col,
        value_final_col=value_final_col,
        indicator_ids=indicator_ids,
    )

    opportunity_count = int(len(eligible_firm_years_df) * len(indicator_index))
    extracted_count = int(observations_df["extracted_present"].sum())
    final_count = int(observations_df["final_present"].sum())

    return {
        "n_fy_ok": int(len(eligible_firm_years_df)),
        "k_indicators": int(len(indicator_index)),
        "n_possible": opportunity_count,
        "n_extracted": extracted_count,
        "n_no_evidence": int(opportunity_count - extracted_count),
        "n_final": final_count,
        "n_not_final": int(extracted_count - final_count),
    }


def plot_indicator_opportunity_sankey(
    opportunity_counts: dict[str, int],
    *,
    title: str = "Indicator-level coverage analysis",
) -> go.Figure:
    possible_count = int(opportunity_counts["n_possible"])
    extracted_count = int(opportunity_counts["n_extracted"])
    no_evidence_count = int(opportunity_counts["n_no_evidence"])
    final_count = int(opportunity_counts["n_final"])
    not_final_count = int(opportunity_counts["n_not_final"])

    figure = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    label=[
                        f"<b>Indicator-company-year observations</b><br>{possible_count:,}",
                        f"<b>Disclosure detected</b><br>{extracted_count:,}",
                        f"<b>No disclosure detected</b><br>{no_evidence_count:,}",
                        f"<b>Value standardized</b><br>{final_count:,}",
                        f"<b>Not standardizable</b><br>{not_final_count:,}",
                    ],
                    x=[0.01, 0.50, 0.50, 0.99, 0.99],
                    y=[0.50, 0.30, 0.70, 0.30, 0.70],
                    pad=22,
                    thickness=18,
                    color=[
                        "cornflowerblue",
                        "cadetblue",
                        "darkgrey",
                        "darkseagreen",
                        "darkgrey",
                    ],
                ),
                link=dict(
                    source=[0, 0, 1, 1],
                    target=[1, 2, 3, 4],
                    value=[extracted_count, no_evidence_count, final_count, not_final_count],
                    color=[
                        to_rgba("cornflowerblue", 0.35),
                        to_rgba("darkgrey", 0.35),
                        to_rgba("cadetblue", 0.35),
                        to_rgba("darkgrey", 0.35),
                    ],
                ),
            )
        ]
    )

    figure.update_layout(
        title=title,
        hovermode="x",
        font=dict(size=15),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=420,
        width=980,
        margin=dict(l=20, r=20, t=50, b=70),
    )
    figure.show()
    return figure


def compute_indicator_extractability_counts(
    firm_stage_df: pd.DataFrame,
    indicator_df: pd.DataFrame,
    *,
    firm_col_firm_stage: str = "firm",
    year_col_firm_stage: str = "year",
    chunks_col_firm_stage: str = "chunks_present",
    firm_col_indicator: str = "company_id",
    year_col_indicator: str = "year",
    indicator_col_indicator: str = "data_point_id",
    model_output_col: str = "model_output",
    value_final_col: str = "value_final",
    indicator_ids: Optional[list[str]] = None,
) -> pd.DataFrame:
    eligible_firm_years_df, indicator_index, observations_df = prepare_indicator_observations(
        firm_stage_df,
        indicator_df,
        firm_col_firm_stage=firm_col_firm_stage,
        year_col_firm_stage=year_col_firm_stage,
        chunks_col_firm_stage=chunks_col_firm_stage,
        firm_col_indicator=firm_col_indicator,
        year_col_indicator=year_col_indicator,
        indicator_col_indicator=indicator_col_indicator,
        model_output_col=model_output_col,
        value_final_col=value_final_col,
        indicator_ids=indicator_ids,
    )

    eligible_counts_df = (
        eligible_firm_years_df.groupby(year_col_firm_stage)
        .size()
        .rename("n_possible")
        .reset_index()
        .rename(columns={year_col_firm_stage: year_col_indicator})
    )

    count_df = (
        observations_df.groupby([year_col_indicator, indicator_col_indicator])[["extracted_present", "final_present"]]
        .sum()
        .rename(columns={"extracted_present": "n_extracted", "final_present": "n_final"})
        .reset_index()
    )

    year_values = np.sort(eligible_counts_df[year_col_indicator].astype(int).unique())
    grid_df = pd.MultiIndex.from_product(
        [year_values, indicator_index],
        names=[year_col_indicator, indicator_col_indicator],
    ).to_frame(index=False)

    out_df = grid_df.merge(eligible_counts_df, on=year_col_indicator, how="left").merge(
        count_df,
        on=[year_col_indicator, indicator_col_indicator],
        how="left",
    )
    out_df["n_possible"] = out_df["n_possible"].fillna(0).astype(int)
    out_df["n_extracted"] = out_df["n_extracted"].fillna(0).astype(int)
    out_df["n_final"] = out_df["n_final"].fillna(0).astype(int)

    return (
        out_df.groupby(indicator_col_indicator, as_index=False)[["n_possible", "n_extracted", "n_final"]]
        .sum()
        .sort_values(indicator_col_indicator)
        .reset_index(drop=True)
    )


def load_indicator_metadata(indicator_metadata_path: Path) -> pd.DataFrame:
    indicator_metadata_df = pd.read_csv(indicator_metadata_path)
    if "id" not in indicator_metadata_df.columns:
        indicator_metadata_df = pd.read_csv(indicator_metadata_path, delimiter=";")

    required_columns = {"id", "label"}
    missing_columns = required_columns - set(indicator_metadata_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required indicator metadata columns: {missing_columns}")

    if "label_specification" in indicator_metadata_df.columns:
        description = indicator_metadata_df["label_specification"].fillna("").astype(str).str.strip()
    elif "specification" in indicator_metadata_df.columns:
        label = indicator_metadata_df["label"].fillna("").astype(str).str.strip()
        specification = indicator_metadata_df["specification"].fillna("").astype(str).str.strip()
        description = label.where(specification.eq(""), label + ": " + specification)
    else:
        description = indicator_metadata_df["label"].fillna("").astype(str).str.strip()

    indicator_metadata_df["datapoint_description"] = description.replace("", pd.NA)
    return indicator_metadata_df


def build_indicator_extractability_table(
    firm_stage_df: pd.DataFrame,
    indicator_df: pd.DataFrame,
    indicator_metadata_df: pd.DataFrame,
) -> pd.DataFrame:
    extractability_df = compute_indicator_extractability_counts(firm_stage_df, indicator_df)
    extractability_df["disclosure_rate"] = np.where(
        extractability_df["n_possible"] > 0,
        extractability_df["n_extracted"] / extractability_df["n_possible"],
        np.nan,
    )
    extractability_df["extraction_success"] = np.where(
        extractability_df["n_extracted"] > 0,
        extractability_df["n_final"] / extractability_df["n_extracted"],
        np.nan,
    )
    extractability_df["overall_yield"] = np.where(
        extractability_df["n_possible"] > 0,
        extractability_df["n_final"] / extractability_df["n_possible"],
        np.nan,
    )

    for column_name in ["disclosure_rate", "extraction_success", "overall_yield"]:
        extractability_df[column_name] = extractability_df[column_name].clip(0, 1)

    metadata_columns = indicator_metadata_df[["id", "datapoint_description"]].drop_duplicates()
    extractability_df = extractability_df.merge(
        metadata_columns,
        left_on="data_point_id",
        right_on="id",
        how="left",
    ).drop(columns="id")

    return extractability_df.sort_values(
        ["datapoint_description", "data_point_id"],
        na_position="last",
    ).reset_index(drop=True)


def latex_escape(text: object) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    escaped = "" if text is None else str(text)
    for source, target in replacements.items():
        escaped = escaped.replace(source, target)
    return escaped


def render_indicator_coverage_longtable(
    indicator_extractability_df: pd.DataFrame,
    *,
    caption: str = "Per-indicator disclosure and extraction coverage.",
    label: str = "tab:indicator_coverage",
    max_rows: Optional[int] = None,
    notes: Optional[str] = None,
) -> str:
    required_columns = {"datapoint_description", "disclosure_rate", "extraction_success"}
    missing_columns = required_columns - set(indicator_extractability_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    table_df = indicator_extractability_df[
        ["datapoint_description", "disclosure_rate", "extraction_success"]
    ].copy()
    table_df["sort_key"] = table_df["datapoint_description"].fillna("").astype(str).str.lower()
    table_df = table_df.sort_values("sort_key", kind="mergesort").drop(columns="sort_key")

    if max_rows is not None:
        table_df = table_df.head(max_rows)

    def format_percent(value: float) -> str:
        if pd.isna(value):
            return r"\textemdash"
        return f"{float(value) * 100:.1f}"

    body = "\n".join(
        f"{latex_escape(row.datapoint_description)} & {format_percent(row.disclosure_rate)} & {format_percent(row.extraction_success)} \\\\"  # noqa: E501
        for row in table_df.itertuples(index=False)
    )

    notes_text = notes or (
        "Disclosure rate is n_extracted / n_possible. Extraction rate is "
        "n_final / n_extracted. Rates are reported as percentages."
    )

    return rf"""\begin{{ThreePartTable}}
\fontsize{{9pt}}{{9pt}}\selectfont\renewcommand{{\arraystretch}}{{1.2}}

\begin{{TableNotes}}[flushleft]
\fontsize{{10pt}}{{10pt}}\linespread{{1.2}}\selectfont
\item \textbf{{Notes:}} {latex_escape(notes_text)}
\end{{TableNotes}}

\begin{{longtable}}{{p{{12cm}}rr}}
\caption{{{latex_escape(caption)}}}\label{{{label}}}\\
\toprule
Indicator & Disclosure rate (\%) & Extraction rate (\%) \\
\midrule
\endfirsthead

\toprule
Indicator & Disclosure rate (\%) & Extraction rate (\%) \\
\midrule
\endhead

\bottomrule
\insertTableNotes
\endfoot

\bottomrule
\endlastfoot

{body}
\end{{longtable}}
\end{{ThreePartTable}}"""


def load_json_payload(json_path: Path) -> dict[str, Any]:
    with json_path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {json_path}")
    return payload



def write_json_payload(payload: Any, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return output_path



def constant_reporters_by_indicator(
    indicator_df: pd.DataFrame,
    years: range = range(2014, 2024),
    company_col: str = "company_id",
    year_col: str = "year",
    indicator_col: str = "data_point_id",
    value_col: str = "value_final",
) -> dict[str, list[str]]:
    required_columns = {company_col, year_col, indicator_col, value_col}
    missing_columns = required_columns - set(indicator_df.columns)
    if missing_columns:
        raise ValueError(f"Missing indicator columns: {missing_columns}")

    reporter_df = indicator_df[[company_col, year_col, indicator_col, value_col]].copy()
    reporter_df[year_col] = reporter_df[year_col].astype(int)
    reporter_df = reporter_df[reporter_df[year_col].isin(list(map(int, years)))]

    reporter_df["has_value"] = reporter_df[value_col].notna()
    reported_df = (
        reporter_df.groupby([indicator_col, company_col, year_col], as_index=False)["has_value"]
        .any()
    )

    complete_df = (
        reported_df[reported_df["has_value"]]
        .groupby([indicator_col, company_col])[year_col]
        .nunique()
        .reset_index(name="n_years_with_value")
    )
    complete_df = complete_df[complete_df["n_years_with_value"] == len(list(map(int, years)))]

    payload = (
        complete_df.groupby(indicator_col)[company_col]
        .apply(lambda company_ids: sorted(set(map(str, company_ids))))
        .to_dict()
    )
    return dict(sorted(payload.items(), key=lambda item: item[0]))



## Data Preparation


In [ ]:
companies_df = pd.read_csv(COMPANIES_PATH)
indicator_df = pd.read_csv(INDICATORS_PATH)
indicator_metadata_df = load_indicator_metadata(INDICATOR_METADATA_PATH)

report_index_df = load_report_index(REPORT_INDEX_PATH)
report_grid_df = build_report_grid(report_index_df)
firm_stage_df = build_firm_stage_table(report_grid_df)

firm_stage_df.head(10)


## Population Coverage


In [ ]:
plot_representativeness_heatmap(
    companies_df,
    firm_stage_df,
    value="coverage_rate",
    by="sector",
    heat_ax_pos=(0.28, 0.15, 0.5, 0.75),
    cbar_ax_pos=(0.80, 0.15, 0.03, 0.75),
    output_path=HEATMAP_OUTPUT_DIR / "heatmap_sector.pdf",
)

plot_representativeness_heatmap(
    companies_df,
    firm_stage_df,
    value="coverage_rate",
    by="country",
    heat_ax_pos=(0.28, 0.15, 0.5, 0.75),
    cbar_ax_pos=(0.80, 0.15, 0.03, 0.75),
    output_path=HEATMAP_OUTPUT_DIR / "heatmap_country.pdf",
)


## Pipeline Coverage


In [ ]:
plot_firm_year_pipeline_sankey(firm_stage_df)
indicator_opportunity_counts = compute_indicator_opportunity_counts(firm_stage_df, indicator_df)
plot_indicator_opportunity_sankey(indicator_opportunity_counts)

indicator_opportunity_counts


## Indicator Extractability


In [ ]:
indicator_extractability_df = build_indicator_extractability_table(
    firm_stage_df,
    indicator_df,
    indicator_metadata_df,
)
indicator_extractability_df.to_csv(INDICATOR_EXTRACTABILITY_OUTPUT_PATH, index=False)
LATEX_OUTPUT_PATH.write_text(
    render_indicator_coverage_longtable(indicator_extractability_df),
    encoding="utf-8",
)

indicator_extractability_df


## Document quality


In [ ]:
_WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)?")

def topk_fourgram_share(text: str, k: int = 5) -> tuple[list[tuple[str, int]], float, int]:
    tokens = [token.lower() for token in _WORD_RE.findall(text)]
    n_tokens = len(tokens)
    if n_tokens < 4:
        return [], 0.0, 0

    counter = Counter()
    for index in range(n_tokens - 3):
        gram = f"{tokens[index]} {tokens[index + 1]} {tokens[index + 2]} {tokens[index + 3]}"
        counter[gram] += 1

    n_4grams_total = n_tokens - 3
    top_k = counter.most_common(k)
    top_k_count = sum(count for _, count in top_k)
    share = top_k_count / n_4grams_total if n_4grams_total else 0.0
    return top_k, share, n_4grams_total

def _parse_company_year(filename: str) -> tuple[Optional[str], Optional[int]]:
    base = os.path.splitext(filename)[0]
    try:
        company_id, year_text = base.rsplit("_", 1)
        return company_id, int(year_text)
    except ValueError:
        return None, None

def process_text_quality_file(text_dir: str, filename: str) -> Optional[dict[str, Any]]:
    if not filename.endswith(".json"):
        return None

    company_id, year = _parse_company_year(filename)
    if company_id is None or year is None:
        return None

    path = os.path.join(text_dir, filename)
    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except (OSError, json.JSONDecodeError):
        return None

    text = payload.get("text", "")
    if not isinstance(text, str):
        return None

    top5_share = topk_fourgram_share(text, k=5)[1]

    return {
        "company_id": company_id,
        "year": year,
        "fog_index": textstat.gunning_fog(text),
        "newline_density": text.count("\n") / len(text) if text else 0.0,
        "top5_4gram_share": top5_share,
    }

text_quality_files = sorted(filename for filename in os.listdir(TEXT_DIR) if filename.endswith(".json"))
max_workers = min(40, (os.cpu_count() or 40))

text_quality_rows: list[dict[str, Any]] = []
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_text_quality_file, str(TEXT_DIR), filename) for filename in text_quality_files]
    for future in as_completed(futures):
        row = future.result()
        if row is not None:
            text_quality_rows.append(row)

df_quality = pd.DataFrame(text_quality_rows).sort_values(["company_id", "year"]).reset_index(drop=True)


In [ ]:
df_text_quality = df_quality.copy()
percentile = 2 / 3

df_text_quality["nl_thr_y"] = df_text_quality.groupby("year")["newline_density"].transform(lambda series: series.quantile(percentile))
df_text_quality["bp_thr_y"] = df_text_quality.groupby("year")["top5_4gram_share"].transform(lambda series: series.quantile(percentile))
df_text_quality["fog_thr_y"] = df_text_quality.groupby("year")["fog_index"].transform(lambda series: series.quantile(percentile))

df_text_quality["bad_newlines"] = (df_text_quality["newline_density"] >= df_text_quality["nl_thr_y"]).astype(int)
df_text_quality["bad_boilerplate"] = (df_text_quality["top5_4gram_share"] >= df_text_quality["bp_thr_y"]).astype(int)
df_text_quality["bad_fog"] = (df_text_quality["fog_index"] >= df_text_quality["fog_thr_y"]).astype(int)

df_text_quality["n_bad_pillars"] = df_text_quality[["bad_newlines", "bad_boilerplate", "bad_fog"]].sum(axis=1)

df_top_quality = df_text_quality[df_text_quality["n_bad_pillars"] == 0].copy()
df_top_quality["company_year"] = df_top_quality["company_id"] + "_" + df_top_quality["year"].astype(str)

top_quality_documents_payload = {
    "top_quality_documents_yearly": df_top_quality["company_year"].tolist()
}

write_json_payload(top_quality_documents_payload, TOP_QUALITY_DOCUMENTS_OUTPUT_PATH)

{
    "top_quality_documents_output": str(TOP_QUALITY_DOCUMENTS_OUTPUT_PATH),
}


## Constant reporters

In [ ]:
constant_reporters_payload = constant_reporters_by_indicator(indicator_df, years=range(2014, 2024))
write_json_payload(constant_reporters_payload, CONSTANT_REPORTERS_OUTPUT_PATH)

{
    "constant_reporters_output": str(CONSTANT_REPORTERS_OUTPUT_PATH),
}
